In [1]:
# jupyter notebook 전체화면으로 변경  
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [1]:
# load module
import os
#os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import copy
import warnings
import torch
import optuna
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import pytorch_lightning as pl
import seaborn as sns

from my_funs import *
from plot_round_cut import *


from itertools import product
from sklearn import metrics
from matplotlib import gridspec
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger


from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet , EncoderNormalizer , GroupNormalizer
from pytorch_forecasting.metrics import SMAPE, PoissonLoss, QuantileLoss
#from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters

#import tensorflow as tf 
import tensorboard as tb 
#tf.io.gfile = tb.compat.tensorflow_stub.io.gfile

import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")  # avoid printing out absolute paths
plt.rcParams['font.family'] = 'NanumGothic'

dongs = ['강남동', '교  동', '근화동', '남  면', '남산면', '동  면', '동내면', '동산면', '북산면',
       '사북면', '서  면', '석사동', '소양동', '신동면', '신북읍', '신사우동', '약사명동', '조운동',
       '퇴계동', '효자1동', '후평1동']

/home/nplab/.local/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: /home/nplab/.local/lib/python3.8/site-packages/torchvision/image.so: undefined symbol: _ZN2at4_ops19empty_memory_format4callEN3c108ArrayRefIlEENS2_8optionalINS2_10ScalarTypeEEENS5_INS2_6LayoutEEENS5_INS2_6DeviceEEENS5_IbEENS5_INS2_12MemoryFormatEEE
  warn(f"Failed to load image Python extension: {e}")
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.1.36ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will n

In [2]:
# batch 2
setting = pd.read_csv('batch_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[0]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 100 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
13,2,0.5,0.120440
4,1,0.5,0.120440
22,3,0.5,0.120440
3,1,0.4,0.101810
21,3,0.4,0.101810
12,2,0.4,0.101810
2,1,0.3,0.096089
20,3,0.3,0.096089
11,2,0.3,0.096089
1,1,0.2,0.091173


In [3]:
# batch 4
setting = pd.read_csv('batch_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[1]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 100 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
13,2,0.5,0.117759
4,1,0.5,0.117759
22,3,0.5,0.117759
2,1,0.3,0.095109
20,3,0.3,0.095109
11,2,0.3,0.095109
3,1,0.4,0.090976
21,3,0.4,0.090976
12,2,0.4,0.090976
1,1,0.2,0.089184


In [5]:
# batch 6
setting = pd.read_csv('batch_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[2]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 100 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
5,1,0.6,0.131058
23,3,0.6,0.131058
14,2,0.6,0.131058
13,2,0.5,0.129815
22,3,0.5,0.129815
4,1,0.5,0.129815
21,3,0.4,0.122676
3,1,0.4,0.122676
12,2,0.4,0.122676
11,2,0.3,0.113158


In [6]:
# batch 8
setting = pd.read_csv('batch_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[3]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 100 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
2,1,0.3,0.111395
20,3,0.3,0.111395
11,2,0.3,0.111395
3,1,0.4,0.097731
21,3,0.4,0.097731
12,2,0.4,0.097731
1,1,0.2,0.097673
19,3,0.2,0.097673
10,2,0.2,0.097673
0,1,0.1,0.084039


In [7]:
# batch 12
setting = pd.read_csv('batch_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[4]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 100 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
13,2,0.5,0.111848
4,1,0.5,0.111848
22,3,0.5,0.111848
5,1,0.6,0.104554
23,3,0.6,0.104554
14,2,0.6,0.104554
2,1,0.3,0.099770
20,3,0.3,0.099770
11,2,0.3,0.099770
3,1,0.4,0.099676


In [2]:
# batch 16
setting = pd.read_csv('batch_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[5]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 100 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
2,1,0.3,0.109579
20,3,0.3,0.109579
11,2,0.3,0.109579
3,1,0.4,0.106468
21,3,0.4,0.106468
12,2,0.4,0.106468
1,1,0.2,0.095089
19,3,0.2,0.095089
10,2,0.2,0.095089
0,1,0.1,0.088409


In [2]:
# batch 20
setting = pd.read_csv('batch_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[6]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 100 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
2,1,0.3,0.113153
20,3,0.3,0.113153
11,2,0.3,0.113153
1,1,0.2,0.102398
19,3,0.2,0.102398
10,2,0.2,0.102398
0,1,0.1,0.089528
18,3,0.1,0.089528
9,2,0.1,0.089528
3,1,0.4,0.087179


In [3]:
# batch 24
setting = pd.read_csv('batch_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[7]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 100 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
3,1,0.4,0.113139
21,3,0.4,0.113139
12,2,0.4,0.113139
2,1,0.3,0.113005
20,3,0.3,0.113005
11,2,0.3,0.113005
1,1,0.2,0.103799
19,3,0.2,0.103799
10,2,0.2,0.103799
13,2,0.5,0.099119


In [4]:
# batch 28
setting = pd.read_csv('batch_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[8]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 100 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
13,2,0.5,0.118679
4,1,0.5,0.118679
22,3,0.5,0.118679
3,1,0.4,0.118299
12,2,0.4,0.118299
21,3,0.4,0.118299
2,1,0.3,0.115847
11,2,0.3,0.115847
20,3,0.3,0.115847
5,1,0.6,0.108827


In [5]:
# batch 32
setting = pd.read_csv('batch_setting.csv')

funs = moving_average_alpha
factor = 0.2
cut_off = [1, 2, 3]
round_cut = np.arange(1,10)/ 10

f2_df = pd.DataFrame(product(cut_off, round_cut) , columns = ['cut_off' , 'round_cut'])

model_ckpt = setting['model_ckpt'].loc[9]
train_data = data_processing('../../train.csv' , factor , funs)
test_data = data_processing('../../test.csv' , factor, funs)
training = get_training(train_data,24, 24*7)
tft = TemporalFusionTransformer.load_from_checkpoint(model_ckpt)

for idx in f2_df.index:
    cut_off = f2_df['cut_off'].loc[idx]
    round_cut = f2_df['round_cut'].loc[idx]
    f2_df.loc[idx, 'f2_score'] = dong_f2_all(training, tft, test_data, cut_off, round_cut)
    #confusion_matrix_plot_round_cut(training, tft ,test_data, cut_off ,'강남동' , f'epoch 100 / cut_off {cut_off} / round_cut {round_cut}' , round_cut)
f2_df.sort_values(by = 'f2_score' ,  ascending=False)

,cut_off,round_cut,f2_score
3,1,0.4,0.105278
21,3,0.4,0.105278
12,2,0.4,0.105278
1,1,0.2,0.104618
19,3,0.2,0.104618
10,2,0.2,0.104618
2,1,0.3,0.104569
20,3,0.3,0.104569
11,2,0.3,0.104569
0,1,0.1,0.090422
